<a href="https://colab.research.google.com/github/i2mmmmm/train_project/blob/main/20230826_ar_c.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LayerNormalization, MultiHeadAttention, Flatten
import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import LSTM, Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler

In [4]:
from google.colab import drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
s30_train = pd.read_csv('/content/drive/My Drive/철도/s30_train.csv')
s40_train = pd.read_csv('/content/drive/My Drive/철도/s40_train.csv')
s50_train = pd.read_csv('/content/drive/My Drive/철도/s50_train.csv')
s70_train = pd.read_csv('/content/drive/My Drive/철도/s70_train.csv')
s100_train = pd.read_csv('/content/drive/My Drive/철도/s100_train.csv')
c30_train = pd.read_csv('/content/drive/My Drive/철도/c30_train.csv')
c40_train = pd.read_csv('/content/drive/My Drive/철도/c40_train.csv')
c50_train = pd.read_csv('/content/drive/My Drive/철도/c50_train.csv')
c70_train = pd.read_csv('/content/drive/My Drive/철도/c70_train.csv')
c100_train = pd.read_csv('/content/drive/My Drive/철도/c100_train.csv')

In [6]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c30_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 31s 145ms/step - loss: 0.0016 - mae: 0.0242 - val_loss: 8.5592e-05 - val_mae: 0.0074
Epoch 2/50
157/157 [==============================] - 15s 94ms/step - loss: 6.6493e-04 - mae: 0.0143 - val_loss: 1.0456e-04 - val_mae: 0.0081
Epoch 3/50
157/157 [==============================] - 17s 109ms/step - loss: 5.0634e-04 - mae: 0.0120 - val_loss: 1.1581e-04 - val_mae: 0.0085
Epoch 4/50
157/157 [==============================] - 15s 97ms/step - loss: 3.9673e-04 - mae: 0.0107 - val_loss: 8.8636e-05 - val_mae: 0.0075
Epoch 5/50
157/157 [==============================] - 15s 97ms/step - loss: 3.6169e-04 - mae: 0.0102 - val_loss: 8.7877e-05 - val_mae: 0.0074
Epoch 6/50
157/157 [==============================] - 15s 98ms/step - loss: 3.5947e-04 - mae: 0.0098 - val_loss: 8.1539e-05 - val_mae: 0.0071
Epoch 7/50
157/157 [==============================] - 15s 93ms/step - loss: 3.1624e-04 - mae: 0.0095 - val_loss: 8.5793e-05 - val_mae: 0.0073
Epoch 8/

In [7]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c30_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 25s 104ms/step - loss: 0.0017 - mae: 0.0279 - val_loss: 6.3669e-04 - val_mae: 0.0202
Epoch 2/50
157/157 [==============================] - 17s 109ms/step - loss: 6.0279e-04 - mae: 0.0167 - val_loss: 3.6668e-04 - val_mae: 0.0153
Epoch 3/50
157/157 [==============================] - 19s 119ms/step - loss: 4.4352e-04 - mae: 0.0142 - val_loss: 4.2459e-04 - val_mae: 0.0165
Epoch 4/50
157/157 [==============================] - 15s 94ms/step - loss: 3.8147e-04 - mae: 0.0131 - val_loss: 3.7247e-04 - val_mae: 0.0154
Epoch 5/50
157/157 [==============================] - 15s 92ms/step - loss: 3.5565e-04 - mae: 0.0124 - val_loss: 2.7232e-04 - val_mae: 0.0131
Epoch 6/50
157/157 [==============================] - 15s 93ms/step - loss: 3.2151e-04 - mae: 0.0119 - val_loss: 2.3449e-04 - val_mae: 0.0123
Epoch 7/50
157/157 [==============================] - 15s 94ms/step - loss: 3.2567e-04 - mae: 0.0116 - val_loss: 2.2827e-04 - val_mae: 0.0119
Epoch 8

In [8]:
predictions_30c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_30c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c40_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

Epoch 1/50
157/157 [==============================] - 26s 108ms/step - loss: 8.8474e-04 - mae: 0.0199 - val_loss: 8.4783e-05 - val_mae: 0.0073
Epoch 2/50
157/157 [==============================] - 19s 124ms/step - loss: 2.0471e-04 - mae: 0.0102 - val_loss: 9.5886e-05 - val_mae: 0.0077
Epoch 3/50
157/157 [==============================] - 16s 100ms/step - loss: 1.4616e-04 - mae: 0.0084 - val_loss: 8.3879e-05 - val_mae: 0.0072
Epoch 4/50
157/157 [==============================] - 15s 97ms/step - loss: 1.2224e-04 - mae: 0.0077 - val_loss: 6.6063e-05 - val_mae: 0.0064
Epoch 5/50
157/157 [==============================] - 15s 97ms/step - loss: 1.0748e-04 - mae: 0.0071 - val_loss: 6.1120e-05 - val_mae: 0.0061
Epoch 6/50
157/157 [==============================] - 18s 117ms/step - loss: 8.8297e-05 - mae: 0.0067 - val_loss: 7.8487e-05 - val_mae: 0.0070
Epoch 7/50
157/157 [==============================] - 15s 98ms/step - loss: 8.8881e-05 - mae: 0.0065 - val_loss: 5.1508e-05 - val_mae: 0.0057
Ep

In [ ]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c40_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

In [ ]:
predictions_40c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_40c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c50_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

In [ ]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c50_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

In [ ]:
predictions_50c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_50c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c70_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

In [ ]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c70_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

In [ ]:
predictions_70c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_70c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
def prepare_data(filename):
    data = filename
    X = data.drop(columns=['Curvature', 'Vertical offset', 'Cross level offset', 'A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W2_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W2_Y_A_axle_L', 'A__B1_W1_Z_A_axle_R'])
    y = data[["YL_M1_B1_W1", "YR_M1_B1_W1"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c100_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 27

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s = model_deepar.predict(X_test_s)

In [ ]:


def prepare_data(filename):
    data = filename
    X = data.drop(columns=['A_M1_B1_W1_Z_L', 'A_M1_B1_BC_Z_L', 'A_M1_B1_W1_Z_R', 'A_M1_B1_BC_Z_R', 'A_M1_B1_W2_Z_R', 'A__B1_W1_Z_A_axle_L', 'A__B1_W1_Z_A_axle_R', 'A__B1_W2_Z_A_axle_R', 'V_M1_B1_W1_L', 'V_M1_B1_W1_R', 'QL_M1_B1_W1', 'QR_M1_B1_W1'])
    y = data[["YL_M1_B1_W2", "YR_M1_B1_W2"]]

    # 데이터 스케일링
    scaler_X = StandardScaler()
    X = scaler_X.fit_transform(X)

    # scaler_y = StandardScaler()  # y 데이터에 대한 새로운 스케일러
    # y = scaler_y.fit_transform(y)  # 2차원 배열로 변환

    return X, y

def reshape_data(X, y, N_TIMESTEPS):
    X_list, y_list = [], []
    X_padded = np.vstack([np.zeros((N_TIMESTEPS, X.shape[1])), X])

    for i in range(len(X_padded) - N_TIMESTEPS):
        X_list.append(X_padded[i:i+N_TIMESTEPS])
        y_list.append(y.iloc[i])

    return np.array(X_list), np.array(y_list)


data_s_X, data_s_y = prepare_data(c100_train)   #####################################################

N_TIMESTEPS = 10
N_FEATURES = 26

X_s, y_s = reshape_data(data_s_X, data_s_y, N_TIMESTEPS)

# 트레이닝/테스트 데이터 나누기
X_train_s = X_s[:10001]
y_train_s = y_s[:10001]
X_test_s = X_s[10001:]
y_test_s = y_s[10001:]


# Early stopping
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=20)

from tensorflow.keras.layers import RepeatVector, TimeDistributed

# DeepAR 스타일의 모델 구축
inputs = tf.keras.Input(shape=(N_TIMESTEPS, N_FEATURES))

# LSTM을 통한 인코딩
x = LSTM(64, return_sequences=True)(inputs)
x = LSTM(128, return_sequences=False)(x)
x = Dropout(0.5)(x)

# 미래 타임스텝에 대한 예측을 위한 RepeatVector
x = RepeatVector(N_TIMESTEPS)(x)

# LSTM을 통한 디코딩
x = LSTM(128, return_sequences=True)(x)
x = LSTM(64, return_sequences=True)(x)
x = Dropout(0.5)(x)

# 각 타임스텝에 대한 출력 값 생성
x = TimeDistributed(Dense(2))(x)

model_deepar = Model(inputs=inputs, outputs=x[:, -1, :])  # 마지막 타임스텝의 출력만 사용

model_deepar.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model_deepar.fit(X_train_s, y_train_s, epochs=50, batch_size=64, validation_data=(X_test_s, y_test_s), callbacks=[es])


# 예측
predictions_s2 = model_deepar.predict(X_test_s)

In [ ]:
predictions_100c = pd.DataFrame(predictions_s, columns=['YL_M1_B1_W1', 'YR_M1_B1_W1'])
predictions_100c2 = pd.DataFrame(predictions_s2, columns=['YL_M1_B1_W2', 'YR_M1_B1_W2'])

In [ ]:
answer_sample = pd.read_csv('/content/drive/My Drive/철도/answer_sample.csv')
c30 = pd.concat([predictions_30c,predictions_30c2],axis=1)
c40 = pd.concat([predictions_40c,predictions_40c2],axis=1)
c50 = pd.concat([predictions_50c,predictions_50c2],axis=1)
c70 = pd.concat([predictions_70c,predictions_70c2],axis=1)
c100 = pd.concat([predictions_100c,predictions_100c2],axis=1)

In [ ]:
c30  = pd.DataFrame(c30, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c40  = pd.DataFrame(c40, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c50  = pd.DataFrame(c50, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c70  = pd.DataFrame(c70, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])
c100  = pd.DataFrame(c100, columns=["YL_M1_B1_W1", "YR_M1_B1_W1", "YL_M1_B1_W2", "YR_M1_B1_W2"])

In [ ]:
c30.columns = ['YL_M1_B1_W1_c30', 'YR_M1_B1_W1_c30', 'YL_M1_B1_W2_c30', 'YR_M1_B1_W2_c30']
c40.columns = ['YL_M1_B1_W1_c40', 'YR_M1_B1_W1_c40', 'YL_M1_B1_W2_c40', 'YR_M1_B1_W2_c40']
c50.columns = ['YL_M1_B1_W1_c50', 'YR_M1_B1_W1_c50', 'YL_M1_B1_W2_c50', 'YR_M1_B1_W2_c50']
c70.columns = ['YL_M1_B1_W1_c70', 'YR_M1_B1_W1_c70', 'YL_M1_B1_W2_c70', 'YR_M1_B1_W2_c70']
c100.columns = ['YL_M1_B1_W1_c100', 'YR_M1_B1_W1_c100', 'YL_M1_B1_W2_c100', 'YR_M1_B1_W2_c100']

In [ ]:
answer2 = pd.concat([answer_sample.Distance,c30,c40,c50,c70,c100], axis=1)

In [ ]:
answer2

In [ ]:
answer2.to_csv('/content/drive/My Drive/철도/answer20230826_ar_c.csv', index=False)